# Sleep Stage Large Model - `timm` ConvNeXt + Transformer

Large-model implementation for the sleep-stage competition. It keeps the reference notebook's 480-row segment and FFT idea, but replaces the hand-written CNN-LSTM with:

- GPU log-FFT image generation from each `480 x 8` segment
- pretrained `timm` ConvNeXt-Large image encoder
- Transformer encoder over consecutive 30-second segment embeddings
- grouped validation by source file
- weighted F1 monitoring and Kaggle submission generation

Designed for a GPU budget around 40 GB VRAM. Reduce `IMG_SIZE`, `SEQ_LEN`, `ENCODER_CHUNK`, or use `convnext_base` if memory is tight.

## Setup

In [ ]:
import math
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print("device:", DEVICE)
print("amp dtype:", AMP_DTYPE)

## Config

In [ ]:
WINDOW_SIZE = 480  # 30 seconds * 16 Hz
SIGNAL_COLUMNS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
LABEL_TO_ID = {"W": 0, "R": 1, "N1": 2, "N2": 3, "N3": 4}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

# Large-model defaults for about 40 GB VRAM.
IMG_SIZE = 384
SEQ_LEN = 32
TRAIN_STRIDE = 8
VALID_STRIDE = 32
BATCH_SIZE = 1
ENCODER_CHUNK = 8
GRAD_ACCUM_STEPS = 4
EPOCHS = 20
PATIENCE = 5
FREEZE_EPOCHS = 2

HEAD_LR = 1e-3
ENCODER_LR = 2e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

TIMM_MODEL_CANDIDATES = [
    "convnext_large.fb_in22k_ft_in1k_384",
    "convnext_large.fb_in22k_ft_in1k",
    "convnext_large_mlp.clip_laion2b_soup_ft_in12k_in1k_384",
    "convnext_large.dinov3_lvd1689m",
    "convnext_base.fb_in22k_ft_in1k",
]

## Paths

In [ ]:
def find_competition_root():
    candidates = [
        Path("/kaggle/input/super-ai-engineer-ss-6-individual-sleep-stage-classification"),
        Path("/kaggle/input/individual-sleep-stage-classification"),
        Path("/kaggle/input/Individual-Sleep-Stage-Classification"),
        Path("/kaggle/input/spai-signal-sleep-staging-classification"),
        Path("../../dataset/super-ai-engineer-ss-6-individual-sleep-stage-classification"),
        Path("../../dataset/individual-sleep-stage-classification"),
        Path("../../dataset/Individual-Sleep-Stage-Classification"),
        Path("../../dataset/sleep_stage_probe"),
    ]
    for root in candidates:
        if root.exists():
            return root
    raise FileNotFoundError("Competition data root not found. Update COMPETITION_ROOT manually.")


def first_existing(*paths):
    for path in paths:
        if path.exists():
            return path
    return paths[0]


COMPETITION_ROOT = find_competition_root()
TRAIN_DIR = first_existing(COMPETITION_ROOT / "train" / "train", COMPETITION_ROOT / "train")
TEST_DIR = first_existing(
    COMPETITION_ROOT / "test_segment" / "test_segment",
    COMPETITION_ROOT / "test_segment",
    COMPETITION_ROOT / "test",
)
SAMPLE_SUBMISSION = COMPETITION_ROOT / "sample_submission.csv"

train_files = sorted(TRAIN_DIR.glob("*.csv"))
test_files = sorted(TEST_DIR.glob("**/*.csv"))

print("root:", COMPETITION_ROOT)
print("train files:", len(train_files), TRAIN_DIR)
print("test files:", len(test_files), TEST_DIR)
print("sample submission:", SAMPLE_SUBMISSION)

In [ ]:
if train_files:
    display(pd.read_csv(train_files[0], nrows=3))
if test_files:
    display(pd.read_csv(test_files[0], nrows=3))

## Load 480-Step Raw Segments

In [ ]:
def clean_signal_frame(df):
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    missing = [col for col in SIGNAL_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing signal columns: {missing}")
    signals = df[SIGNAL_COLUMNS].apply(pd.to_numeric, errors="coerce")
    signals = signals.interpolate(limit_direction="both").ffill().bfill().fillna(0.0)
    return signals.astype("float32")


def majority_label(window_labels):
    counts = np.bincount(window_labels, minlength=len(LABEL_TO_ID))
    return int(counts.argmax())


def load_raw_segments(files, is_training=True, window_size=WINDOW_SIZE):
    segments = []
    labels = []
    groups = []
    segment_ids = []

    for file_idx, file_path in enumerate(tqdm(files, desc="Loading segments")):
        df = pd.read_csv(file_path)
        signal = clean_signal_frame(df)
        num_windows = len(signal) // window_size
        if num_windows == 0:
            continue

        values = signal.iloc[: num_windows * window_size].to_numpy(dtype="float32")
        windows = values.reshape(num_windows, window_size, len(SIGNAL_COLUMNS))
        segments.append(windows)
        groups.extend([file_idx] * num_windows)

        if is_training:
            y = df["Sleep_Stage"].map(LABEL_TO_ID).to_numpy()
            y = y[: num_windows * window_size].reshape(num_windows, window_size)
            labels.append(np.apply_along_axis(majority_label, axis=1, arr=y))
        else:
            stem = file_path.stem
            segment_ids.extend([stem] if num_windows == 1 else [f"{stem}_{i:05d}" for i in range(num_windows)])

    X = np.vstack(segments).astype("float32")
    groups = np.asarray(groups)
    if is_training:
        y = np.concatenate(labels).astype("int64")
        return X, y, groups
    return X, segment_ids, groups

In [ ]:
X, y, groups = load_raw_segments(train_files, is_training=True)
print("X:", X.shape)
print("y:", y.shape)
print("groups:", groups.shape)
print("label counts:", {ID_TO_LABEL[k]: v for k, v in Counter(y).items()})

## Grouped Split and Scaling

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, valid_idx = next(splitter.split(X, y, groups=groups))
train_idx = np.sort(train_idx)
valid_idx = np.sort(valid_idx)

X_train, X_valid = X[train_idx], X[valid_idx]
y_train, y_valid = y[train_idx], y[valid_idx]
groups_train, groups_valid = groups[train_idx], groups[valid_idx]

scaler = StandardScaler()
train_shape = X_train.shape
valid_shape = X_valid.shape
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(train_shape).astype("float32")
X_valid = scaler.transform(X_valid.reshape(-1, X_valid.shape[-1])).reshape(valid_shape).astype("float32")

print("train:", X_train.shape, Counter(y_train))
print("valid:", X_valid.shape, Counter(y_valid))

## Sequence Dataset

In [ ]:
def build_sequence_indices(groups_array, seq_len, stride):
    sequences = []
    for group_id in np.unique(groups_array):
        idx = np.where(groups_array == group_id)[0]
        if len(idx) <= seq_len:
            sequences.append(idx)
            continue
        for start in range(0, len(idx) - seq_len + 1, stride):
            sequences.append(idx[start : start + seq_len])
        last = idx[-seq_len:]
        if len(sequences[-1]) == 0 or not np.array_equal(sequences[-1], last):
            sequences.append(last)
    return sequences


class SegmentSequenceDataset(Dataset):
    def __init__(self, X, y, sequences, seq_len=SEQ_LEN):
        self.X = X
        self.y = y
        self.sequences = sequences
        self.seq_len = seq_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        mask = np.zeros(self.seq_len, dtype=bool)
        x = np.zeros((self.seq_len, WINDOW_SIZE, len(SIGNAL_COLUMNS)), dtype="float32")
        y = np.full(self.seq_len, -100, dtype="int64")

        n = min(len(seq), self.seq_len)
        x[:n] = self.X[seq[:n]]
        mask[:n] = True
        if self.y is not None:
            y[:n] = self.y[seq[:n]]
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(mask)


train_sequences = build_sequence_indices(groups_train, SEQ_LEN, TRAIN_STRIDE)
valid_sequences = build_sequence_indices(groups_valid, SEQ_LEN, VALID_STRIDE)

train_ds = SegmentSequenceDataset(X_train, y_train, train_sequences)
valid_ds = SegmentSequenceDataset(X_valid, y_valid, valid_sequences)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("train sequences:", len(train_sequences))
print("valid sequences:", len(valid_sequences))

## Large ConvNeXt + Transformer Model

In [ ]:
def resolve_timm_model(candidates):
    available = set(timm.list_models(pretrained=True))
    for name in candidates:
        if name in available:
            return name
    for name in candidates:
        prefix = name.split(".")[0]
        matches = [m for m in available if m.startswith(prefix)]
        if matches:
            print("fallback matches for", prefix, matches[:10])
            return matches[-1]
    raise ValueError("No candidate timm model found. Check timm version or model names.")


TIMM_MODEL = resolve_timm_model(TIMM_MODEL_CANDIDATES)
print("using timm model:", TIMM_MODEL)

In [ ]:
class LargeSleepStageModel(nn.Module):
    def __init__(self, timm_model, num_classes=5, seq_len=SEQ_LEN, img_size=IMG_SIZE, encoder_chunk=ENCODER_CHUNK):
        super().__init__()
        self.seq_len = seq_len
        self.img_size = img_size
        self.encoder_chunk = encoder_chunk

        self.encoder = timm.create_model(
            timm_model,
            pretrained=True,
            in_chans=3,
            num_classes=0,
            global_pool="avg",
        )
        embed_dim = getattr(self.encoder, "num_features", None)
        if embed_dim is None:
            with torch.no_grad():
                dummy = torch.zeros(1, 3, img_size, img_size)
                embed_dim = self.encoder(dummy).shape[-1]

        nhead = 12 if embed_dim % 12 == 0 else 8
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=embed_dim * 4,
            dropout=0.25,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.pos = nn.Parameter(torch.zeros(1, seq_len, embed_dim))
        self.sequence_model = nn.TransformerEncoder(layer, num_layers=4)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def make_logfft_images(self, x):
        # x: B, S, T, C. Create three spectrogram-like maps from log FFT magnitude.
        b, s, _, _ = x.shape
        spec = torch.fft.rfft(x.float(), dim=2).abs().log1p()
        bvp = spec[..., 0]
        acc = torch.sqrt(torch.clamp((spec[..., 1:4] ** 2).sum(dim=-1), min=1e-8))
        phys = spec[..., 4:8].mean(dim=-1)
        img = torch.stack([bvp, acc, phys], dim=2)  # B, S, 3, F
        img = img.unsqueeze(-1).repeat(1, 1, 1, 1, len(SIGNAL_COLUMNS))
        img = img.reshape(b * s, 3, img.shape[-2], img.shape[-1])
        img = F.interpolate(img, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False)
        mean = img.mean(dim=(2, 3), keepdim=True)
        std = img.std(dim=(2, 3), keepdim=True).clamp_min(1e-5)
        return (img - mean) / std

    def encode_images(self, images):
        outputs = []
        for start in range(0, len(images), self.encoder_chunk):
            outputs.append(self.encoder(images[start : start + self.encoder_chunk]))
        return torch.cat(outputs, dim=0)

    def forward(self, x, mask):
        b, s, _, _ = x.shape
        images = self.make_logfft_images(x)
        emb = self.encode_images(images).reshape(b, s, -1)
        emb = emb + self.pos[:, :s]
        z = self.sequence_model(emb, src_key_padding_mask=~mask.bool())
        z = self.norm(z)
        return self.classifier(z)


model = LargeSleepStageModel(TIMM_MODEL).to(DEVICE)
model

In [ ]:
def set_encoder_trainable(model, trainable):
    for p in model.encoder.parameters():
        p.requires_grad = trainable


class_counts = np.bincount(y_train, minlength=len(LABEL_TO_ID)).astype("float32")
class_weights = class_counts.sum() / np.maximum(class_counts, 1.0)
class_weights = class_weights / class_weights.mean()
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
set_encoder_trainable(model, False)

optimizer = torch.optim.AdamW(
    [
        {"params": [p for n, p in model.named_parameters() if "encoder" not in n and p.requires_grad], "lr": HEAD_LR},
        {"params": [p for n, p in model.named_parameters() if "encoder" in n and p.requires_grad], "lr": ENCODER_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-6)
scaler_amp = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))

print("class weights:", class_weights.detach().cpu().numpy())

## Train

In [ ]:
def make_optimizer(model):
    head_params = []
    encoder_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("encoder."):
            encoder_params.append(param)
        else:
            head_params.append(param)
    return torch.optim.AdamW(
        [
            {"params": head_params, "lr": HEAD_LR},
            {"params": encoder_params, "lr": ENCODER_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )


def run_epoch(loader, train=False):
    model.train(train)
    total_loss = 0.0
    all_pred = []
    all_true = []
    if train:
        optimizer.zero_grad(set_to_none=True)

    for step, (xb, yb, mask) in enumerate(tqdm(loader, leave=False)):
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
                logits = model(xb, mask)
                loss = criterion(logits.reshape(-1, len(LABEL_TO_ID)), yb.reshape(-1))
                loss_for_backward = loss / GRAD_ACCUM_STEPS

            if train:
                scaler_amp.scale(loss_for_backward).backward()
                if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(loader):
                    scaler_amp.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler_amp.step(optimizer)
                    scaler_amp.update()
                    optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * int(mask.sum().item())
        pred = logits.detach().argmax(dim=-1)
        valid = mask.detach().bool()
        all_pred.append(pred[valid].cpu().numpy())
        all_true.append(yb.detach()[valid].cpu().numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    avg_loss = total_loss / max(len(y_true), 1)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    return avg_loss, weighted_f1, y_true, y_pred

In [ ]:
best_path = Path("sleep_stage_convnext_large_transformer.pt")
best_f1 = -1.0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_EPOCHS + 1:
        print("unfreezing timm encoder")
        set_encoder_trainable(model, True)
        optimizer = make_optimizer(model)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-6)

    train_loss, train_f1, _, _ = run_epoch(train_loader, train=True)
    valid_loss, valid_f1, _, _ = run_epoch(valid_loader, train=False)
    scheduler.step(valid_f1)

    print(
        f"epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
        f"valid_loss={valid_loss:.4f} valid_f1={valid_f1:.4f}"
    )

    if valid_f1 > best_f1:
        best_f1 = valid_f1
        bad_epochs = 0
        torch.save({"model": model.state_dict(), "timm_model": TIMM_MODEL, "best_f1": best_f1}, best_path)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("early stopping")
            break

print("best valid weighted F1:", best_f1)

## Validation Report

In [ ]:
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model"])
valid_loss, valid_f1, y_true, y_pred = run_epoch(valid_loader, train=False)
print("valid loss:", valid_loss)
print("valid weighted F1:", valid_f1)
print(classification_report(
    y_true,
    y_pred,
    labels=list(ID_TO_LABEL),
    target_names=[ID_TO_LABEL[i] for i in sorted(ID_TO_LABEL)],
    zero_division=0,
))

## Predict Test

In [ ]:
X_test, test_ids, test_groups = load_raw_segments(test_files, is_training=False)
test_shape = X_test.shape
X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(test_shape).astype("float32")
test_sequences = build_sequence_indices(test_groups, SEQ_LEN, SEQ_LEN)
test_ds = SegmentSequenceDataset(X_test, None, test_sequences)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print("test:", X_test.shape, "test sequences:", len(test_sequences))

In [ ]:
@torch.no_grad()
def predict_test_sequences(loader, sequences, n_items):
    model.eval()
    pred_sum = np.zeros((n_items, len(LABEL_TO_ID)), dtype="float64")
    pred_count = np.zeros(n_items, dtype="float64")

    for batch_idx, (xb, _, mask) in enumerate(tqdm(loader, desc="Predicting")):
        xb = xb.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            logits = model(xb, mask)
            proba = logits.softmax(dim=-1).detach().cpu().numpy()

        start = batch_idx * loader.batch_size
        for row, seq in enumerate(sequences[start : start + loader.batch_size]):
            valid_len = min(len(seq), SEQ_LEN)
            pred_sum[seq[:valid_len]] += proba[row, :valid_len]
            pred_count[seq[:valid_len]] += 1

    pred_count = np.maximum(pred_count, 1.0)
    return pred_sum / pred_count[:, None]


test_proba = predict_test_sequences(test_loader, test_sequences, len(test_ids))
test_pred = test_proba.argmax(axis=1)
test_labels = [ID_TO_LABEL[int(x)] for x in test_pred]
submission = pd.DataFrame({"id": test_ids, "labels": test_labels})

if SAMPLE_SUBMISSION.exists():
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    submission = sample[["id"]].merge(submission, on="id", how="left")
    submission["labels"] = submission["labels"].fillna("W")

submission.to_csv("submission_convnext_large_transformer.csv", index=False)
submission.head()

In [ ]:
submission["labels"].value_counts()